## Strategy Diff Report

This notebook evaluates how far the simulation values are from the targets established in the NDC document

In [1]:
import pandas as pd
import os

In [2]:
SCRIPT_DIR_PATH = os.getcwd()
PARENT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
TABLEAU_DIR_PATH = os.path.join(PARENT_DIR_PATH, "tableau/data")
DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")

In [3]:
tableau_decomposed_df = pd.read_csv(os.path.join(TABLEAU_DIR_PATH, "decomposed_emissions_bulgaria_2022.csv"))
tableau_decomposed_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125238,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125238,0.125238
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125320,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125320,0.125320
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125392,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125392,0.125392
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125449,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125449,0.125449


In [4]:
# Check ippu just to be sure data is correct
tableau_decomposed_df[tableau_decomposed_df["CSC.Sector"] == 'Energy']

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
232,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.415001,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.415001,NaN
233,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.414772,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.414772,NaN
234,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.415659,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.415659,NaN
235,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.416381,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.416381,NaN
236,0.0,0.0,EN - Building:CH4,Energy,EN - Building,0.417646,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.417646,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4544,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.076789,2018,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.076789,NaN
4545,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.080076,2019,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.080076,NaN
4546,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.076704,2020,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.076704,NaN
4547,NaN,NaN,EN - Transportation:N2O,Energy,EN - Transportation,0.081371,2021,N2O,NaN,NaN,Historical,BGR,bulgaria,EDGAR,0.081371,NaN


In [5]:
# Check Energy for WEM
tableau_decomposed_df[(tableau_decomposed_df["CSC.Sector"] == 'Energy') & (tableau_decomposed_df["strategy"] == 'WAM') & (tableau_decomposed_df["Year"] == 2050)].to_clipboard()

In [6]:
# Aggregate by CSC.Sector and Year fields and sum value
agg_df = tableau_decomposed_df.groupby(['CSC.Sector', 'Year', 'strategy'])['value'].sum().reset_index()
agg_df = agg_df.rename(columns={"CSC.Sector": "Subsector"})

# Make column name lowercase
agg_df.columns = agg_df.columns.str.lower()
agg_df.head()

,subsector,year,strategy,value
0,Agriculture,2000,Historical,5.06396
1,Agriculture,2001,Historical,5.03555
2,Agriculture,2002,Historical,5.00713
3,Agriculture,2003,Historical,4.97872
4,Agriculture,2004,Historical,4.95030


In [7]:
agg_df.subsector.unique()

array(['Agriculture', 'CCSQ', 'Energy', 'Industrial Processes',
       'Land Use, Land Use Change, and Forestry', 'Waste'], dtype=object)

In [8]:
# Filter out years outside relevant years
relevant_years = [2050]
agg_df = agg_df[agg_df['year'].isin(relevant_years)]
agg_df.head()

,subsector,year,strategy,value
107,Agriculture,2050,Strategy TX:BASE,7.067093
108,Agriculture,2050,WAM,5.273773
109,Agriculture,2050,WEM,6.431257
194,CCSQ,2050,Strategy TX:BASE,0.000000
195,CCSQ,2050,WAM,-32.389729


In [9]:
# Filter out CCSQ
agg_df = agg_df[~agg_df['subsector'].str.contains("CCSQ")]
agg_df.head()

,subsector,year,strategy,value
107,Agriculture,2050,Strategy TX:BASE,7.067093
108,Agriculture,2050,WAM,5.273773
109,Agriculture,2050,WEM,6.431257
304,Energy,2050,Strategy TX:BASE,77.063533
305,Energy,2050,WAM,7.666164


In [10]:
# Filter out Historical strategy
agg_df = agg_df[~agg_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
agg_df.head()

,subsector,year,strategy,value
108,Agriculture,2050,WAM,5.273773
109,Agriculture,2050,WEM,6.431257
305,Energy,2050,WAM,7.666164
306,Energy,2050,WEM,15.002094
415,Industrial Processes,2050,WAM,1.148486


In [11]:
# Sort it by strategy and year
agg_df = agg_df.sort_values(by=['strategy', 'year'], ascending=[False, True])
agg_df.head()

,subsector,year,strategy,value
109,Agriculture,2050,WEM,6.431257
306,Energy,2050,WEM,15.002094
416,Industrial Processes,2050,WEM,3.410287
526,"Land Use, Land Use Change, and Forestry",2050,WEM,-9.893206
636,Waste,2050,WEM,1.617092


In [12]:
wide = (
    agg_df
    .pivot(
        index="subsector",
        columns=["year", "strategy"],
        values="value"
    )
)

# flatten the MultiIndex columns → "2022_LEP", "2022_WAM", etc.
wide.columns = [f"{year}_{strategy}" for year, strategy in wide.columns]

wide = wide.reset_index()

In [13]:
wide

,subsector,2050_WEM,2050_WAM
0,Agriculture,6.431257,5.273773
1,Energy,15.002094,7.666164
2,Industrial Processes,3.410287,1.148486
3,"Land Use, Land Use Change, and Forestry",-9.893206,-8.345907
4,Waste,1.617092,1.114894


In [15]:
# Load the report template
template_df = pd.read_csv(os.path.join(DATA_DIR_PATH, "strategy_report_template.csv"))
template_df

,subsector,2050_WEM_target,2050_WAM_target
0,Agriculture,6.0,5.8
1,Energy,15.1,0.0
2,Industrial Processes,3.9,0.5
3,"Land Use, Land Use Change, and Forestry",-9.2,-9.2
4,Waste,1.9,1.4


In [16]:
# Merge with the template
report_df = template_df.merge(wide, how='left', on='subsector')
report_df

,subsector,2050_WEM_target,2050_WAM_target,2050_WEM,2050_WAM
0,Agriculture,6.0,5.8,6.431257,5.273773
1,Energy,15.1,0.0,15.002094,7.666164
2,Industrial Processes,3.9,0.5,3.410287,1.148486
3,"Land Use, Land Use Change, and Forestry",-9.2,-9.2,-9.893206,-8.345907
4,Waste,1.9,1.4,1.617092,1.114894


In [17]:
# Add a total row
total_row = pd.DataFrame(report_df.select_dtypes(include='number').sum()).T
total_row['subsector'] = 'Total'
report_df = pd.concat([report_df, total_row], ignore_index=True)
report_df

,subsector,2050_WEM_target,2050_WAM_target,2050_WEM,2050_WAM
0,Agriculture,6.0,5.8,6.431257,5.273773
1,Energy,15.1,0.0,15.002094,7.666164
2,Industrial Processes,3.9,0.5,3.410287,1.148486
3,"Land Use, Land Use Change, and Forestry",-9.2,-9.2,-9.893206,-8.345907
4,Waste,1.9,1.4,1.617092,1.114894
5,Total,17.7,-1.5,16.567524,6.857410


In [19]:
report_df["ae_2050_WEM"] = (report_df["2050_WEM_target"] - report_df["2050_WEM"]).abs()

report_df["ae_2050_WAM"] = (report_df["2050_WAM_target"] - report_df["2050_WAM"]).abs()


In [20]:
report_df

,subsector,2050_WEM_target,2050_WAM_target,2050_WEM,2050_WAM,ae_2050_WEM,ae_2050_WAM
0,Agriculture,6.0,5.8,6.431257,5.273773,0.431257,0.526227
1,Energy,15.1,0.0,15.002094,7.666164,0.097906,7.666164
2,Industrial Processes,3.9,0.5,3.410287,1.148486,0.489713,0.648486
3,"Land Use, Land Use Change, and Forestry",-9.2,-9.2,-9.893206,-8.345907,0.693206,0.854093
4,Waste,1.9,1.4,1.617092,1.114894,0.282908,0.285106
5,Total,17.7,-1.5,16.567524,6.857410,1.132476,8.357410


In [21]:
report_df.to_csv(os.path.join(TABLEAU_DIR_PATH, "strategy_diff_report.csv"), index=False)